In [73]:
%load_ext aiida
%aiida

The aiida extension is already loaded. To reload it, use:
  %reload_ext aiida


Loaded AiiDA DB environment - profile name: bit.

In [77]:
workpath.show_tree()

workflows
├── Hg0.75Cd0.25Te_work_nosym *
├── cd_elemental *
├── cd_elemental_kspacing_0_03 *
├── cd_elemental_kspacing_0_03_sigma_0_2 *
├── cd_elemental_soc_nosym *
├── cdte_primitive *
├── hg_elemental *
├── hg_elemental_k_spacing_0_03 *
├── hg_elemental_k_spacing_0_03_sigma_0_2 *
├── hg_elemental_soc *
├── hg_elemental_soc_nosym *
├── hgte222_V_hg *
├── hgte222_V_hg_rattle *
├── hgte222_V_hg_soc_nosym *
├── hgte222_supercell *
├── hgte222_supercell_soc_nosym *
├── hgte333_V_hg_gam *
├── hgte333_supercell *
├── hgte_primitive *
├── hgte_primitive_2_V_hg_ispin_2_isym0 *
├── hgte_primitive_2_V_hg_nupdown_2_isym0 *
├── hgte_primitive_k_spacing_0_03 *
├── hgte_primitive_k_spacing_0_03_sigma_0_2 *
├── hgte_true_primitive *
├── hgte_true_primitive_333_V_hg *
├── hgte_true_primitive_333_supercell *
├── mct_333_gamma_relax_V_Hg_27 *
├── mct_333_gamma_relax_V_Hg_28 *
├── mct_333_gamma_relax_V_Hg_35 *
├── mct_333_gamma_supercell *
└── te_elemental *



In [132]:
from doped.chemical_potentials import CompetingPhasesAnalyzer, ComputedStructureEntry

from aiida_grouppathx import GroupPathX

def node2entry(node):
    return ComputedStructureEntry(node.outputs.relax.structure.get_pymatgen(),
                       node.outputs.misc['total_energies']['energy_extrapolated'])

basepath = GroupPathX('mct-defect')
elemental_struct_path = GroupPathX('defects/elemental_ref')
workpath = basepath['workflows']


entries = [
    node2entry(workpath['cd_elemental'].node),
    node2entry(workpath['hg_elemental'].node),
    node2entry(workpath['te_elemental'].node),
    node2entry(workpath['hgte_primitive'].node),
    node2entry(workpath['cdte_primitive'].node),
    #node2entry(workpath['mct_333_gamma_supercell'].node),
    node2entry(mct_64_atoms)
]
entries[-1]._energy -=0.0028*64 # Make Hg0.75CdTe

mct = workpath['mct_333_gamma_supercell'].node.outputs.relax.structure.get_pymatgen().composition
ana = CompetingPhasesAnalyzer(mct, entries)

ana.chempots

{'limits': {'CdHg3Te4-Hg-CdTe': {'Cd': -1.66304,
   'Hg': -0.54968,
   'Te': -3.82592},
  'CdHg3Te4-HgTe-Hg': {'Cd': -1.66492, 'Hg': -0.54968, 'Te': -3.82545},
  'CdHg3Te4-Te-CdTe': {'Cd': -1.96595, 'Hg': -0.85258, 'Te': -3.523},
  'CdHg3Te4-HgTe-Te': {'Cd': -1.96735, 'Hg': -0.8521, 'Te': -3.523}},
 'elemental_refs': {'Te': -3.523, 'Hg': -0.54968, 'Cd': -1.20487},
 'limits_wrt_el_refs': {'CdHg3Te4-Hg-CdTe': {'Cd': -0.45817,
   'Hg': 0.0,
   'Te': -0.3029},
  'CdHg3Te4-HgTe-Hg': {'Cd': -0.46005, 'Hg': 0.0, 'Te': -0.30244},
  'CdHg3Te4-Te-CdTe': {'Cd': -0.76108, 'Hg': -0.3029, 'Te': 0.0},
  'CdHg3Te4-HgTe-Te': {'Cd': -0.76248, 'Hg': -0.30244, 'Te': 0.0}}}